**COMPUTING MONTE CARLO OPTIONS PRICING FOR GOLDMAN SACHS OPTIONS**

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

In [4]:
gs = yf.Ticker('GS')
expirations = gs.options
print(expirations[:10])

('2026-09-11', '2026-09-18', '2026-09-25', '2026-10-02', '2026-10-09', '2026-10-16', '2026-10-23', '2026-11-20', '2026-12-18', '2027-01-15')


In [5]:
opt_chain = gs.option_chain('2026-11-20')
calls = opt_chain.calls
puts = opt_chain.puts

print(calls.head())
print(calls.columns.tolist())

      contractSymbol             lastTradeDate  strike  lastPrice     bid  \
0  GS261120C00400000 2026-08-03 19:20:31+00:00   400.0     621.28  637.85   
1  GS261120C00430000 2026-08-03 19:21:32+00:00   430.0     591.75  610.60   
2  GS261120C00450000 2026-09-08 14:21:51+00:00   450.0     583.80  587.90   
3  GS261120C00460000 2026-08-03 19:23:14+00:00   460.0     561.63  579.40   
4  GS261120C00470000 2026-08-25 13:30:02+00:00   470.0     579.05  568.10   

      ask     change  percentChange  volume  openInterest  impliedVolatility  \
0  645.75   0.000000       0.000000     NaN             0           1.111088   
1  614.65   0.000000       0.000000     NaN             0           1.073369   
2  594.25  22.700012       4.045627     4.0             4           0.946656   
3  586.35   0.000000       0.000000     NaN             0           1.009343   
4  574.45   0.000000       0.000000     1.0             0           0.912476   

   inTheMoney contractSize currency  
0        True     

**CURRENT PRICE**

In [6]:
current_price = gs.history(period='1d')['Close'].iloc[-1]
print(current_price)

1038.375


**T**

In [7]:
from datetime import datetime

today = datetime.now()
expiry = datetime(2026, 11, 20)
T = (expiry - today).days / 365
print(T)

0.19726027397260273


**IMPLIED VOLATILITY**

In [8]:
from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    call_price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    return call_price

In [9]:
from scipy.optimize import brentq

def implied_vol_objective(sigma, S, K, T, r, market_price):
    return black_scholes_call(S, K, T, r, sigma) - market_price

implied_vol_calculated = brentq(implied_vol_objective, 0.01, 3.0, args=(current_price, 1040, T, 0.05, 78.00))
print(implied_vol_calculated)

0.402388413778551


**STIMULATIONG OPTIONS PRICE**

In [10]:
np.random.seed(42)
num_simulations = 10000
Z = np.random.normal(0, 1, num_simulations)

S0 = current_price
sigma = implied_vol_calculated
K = 1040
r = 0.05

simulated_prices = S0 * np.exp((r - sigma**2/2)*T + sigma*np.sqrt(T)*Z)
print(simulated_prices[:10])
print(simulated_prices.shape)

[1127.85918457 1006.86346302 1158.70495333 1354.9182583   989.75586438
  989.75876846 1368.59130044 1183.7692334   948.99402483 1137.13818026]
(10000,)


**PAYOFF COMPUTATION**

In [11]:
payoffs = np.maximum(simulated_prices - K, 0)
print(payoffs[:10])

[ 87.85918457   0.         118.70495333 314.9182583    0.
   0.         328.59130044 143.7692334    0.          97.13818026]


**AVERAGE AND DISCOUNT**

In [12]:
average_payoff = payoffs.mean()
monte_carlo_price = average_payoff * np.exp(-r*T)
print(f"Monte Carlo price: ${monte_carlo_price:.2f}")
print(f"Black-Scholes price: $78.00")

Monte Carlo price: $78.01
Black-Scholes price: $78.00
